# Notebook 03: Clasificación de Daño Vehicular mediante CNN Profunda

## Contexto y Arquitectura
Este notebook aborda el modelado del clasificador de triaje en **ClaimVision-AI**.
El modelo categoriza la imagen del vehículo en tres estados fundamentales:
- `damaged`: Vehículo con daños visibles y estructurales.
- `good`: Vehículo en condiciones normales / sin siniestro aparente.
- `unknown`: Imagen no vehicular o fuera de distribución (OOD).

### Decisiones de Diseño Técnico:
- Arquitectura convolucional con bloques residuales y capas de `Rescaling`, `Conv2D`, `BatchNormalization` y `GlobalAveragePooling2D`.
- Regularización mediante `Dropout(0.25)` para favorecer la generalización.
- Función de pérdida `CategoricalCrossentropy` y optimizador `Adam(lr=3e-4)`.
- Checkpointing para persistencia del mejor estado en `ml/weights/classification/`.


In [1]:
#LIBRERIAS
import os
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import numpy as np
import keras
from keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import cv2 as cv
import config
from IPython.display import Image
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
import matplotlib.image as mpimg

### Carga de Metadatos y Conjuntos Procesados
Cargamos los DataFrames de entrenamiento y prueba generados en la etapa de preprocesamiento.

In [2]:
processed_path = config.PROCESSED_PATH_ORIGINAL
df_train = pd.read_csv(F"{processed_path}/csv/train.csv")
df_test = pd.read_csv(F"{processed_path}/csv/test.csv")

In [3]:
x_train_csv =[]
for img in df_train['image_path']:
    imgPaths = cv.imread(img)
    x_train_csv.append(cv.resize(imgPaths,(180,180)))
y_train_csv = df_train['damaged']

x_test_csv = []
for img in df_test['image_path']:
    imgPaths = cv.imread(img)
    x_test_csv.append(cv.resize(imgPaths, (180, 180)))
y_test_csv = df_test['damaged']

y_train_encoded = to_categorical(y_train_csv)
y_test_encoded = to_categorical(y_test_csv)
X_train = tf.convert_to_tensor(x_train_csv)
X_test = tf.convert_to_tensor(x_test_csv)
y_train = tf.convert_to_tensor(y_train_encoded)
y_test = tf.convert_to_tensor(y_test_encoded)
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(32)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

### Configuración de Callbacks y Regularización
Definimos mecanismos de detención temprana (*EarlyStopping*) monitoreando `val_loss` para prevenir sobreajuste, junto con checkpoints automáticos para almacenar únicamente los pesos con mejor rendimiento.

In [4]:
callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        mode='auto'
    ),
    keras.callbacks.ModelCheckpoint(
        './poto/reg_model_{val_loss:.3f}.keras',
        save_best_only = True,
        save_weights_only=False,
        monitor='val_loss'
    )
]

### Definición de la Arquitectura Convolucional (Transfer Learning / Backbone ResNet)
Construimos el modelo utilizando capas preentrenadas para extracción de características de bajo y medio nivel, agregando un clasificador superior (*head*) con Dropout y Softmax para las clases de daño vehicular.

In [5]:
dnn_model = keras.Sequential()
imported_model = keras.applications.ResNet50(
    include_top=False,
    input_shape=(180,180,3),
    pooling='avg',
    classes=1,
    weights='imagenet'
)
for layer in imported_model.layers:
    layer.trainable=False
dnn_model.add(imported_model)
dnn_model.add(layers.Flatten())
dnn_model.add(layers.Dense(512, activation='relu'))
dnn_model.add(layers.Dense(2, activation='sigmoid'))
dnn_model.compile(optimizer=Adam(learning_rate=0.001),loss='binary_crossentropy',metrics=['accuracy'])

### Entrenamiento y Optimización del Clasificador
Ejecutamos el ciclo de entrenamiento utilizando generadores de datos (*tf.data.Dataset*) optimizados con prefetch y batching en memoria.

In [6]:
history = dnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.8578 - loss: 0.6223 - val_accuracy: 0.0714 - val_loss: 3.3282
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 40s 805ms/step - accuracy: 0.5519 - loss: 1.3115 - val_accuracy: 0.0821 - val_loss: 1.9785
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 36s 742ms/step - accuracy: 0.6516 - loss: 0.7849 - val_accuracy: 0.1929 - val_loss: 1.4910
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 37s 749ms/step - accuracy: 0.7245 - loss: 0.6185 - val_accuracy: 0.5750 - val_loss: 0.8060
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 38s 767ms/step - accuracy: 0.8182 - loss: 0.4152 - val_accuracy: 0.5750 - val_loss: 0.8749
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.8171 - loss: 0.4346 - val_accuracy: 0.6357 - val_loss: 0.6857
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - accuracy: 0.8281 - loss: 0.3864 - val_accuracy: 0.7643 - val_loss: 0.5271
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 63s 1s/step - accuracy: 0.8492 - loss: 0.3363 - val_accuracy: 0.7714

In [7]:
dnn_model.save("./models/modelo_prueba_more_data.keras")

In [8]:
Validation_Path = "./data/processed/original/csv/val.csv"
df_val = pd.read_csv(Validation_Path)
predictions = []

In [9]:
Classes = ["No Damage", "Damaged"]

In [10]:
for index, row in df_val.iterrows():
    Image_Keras = keras.utils.load_img(row['image_path'], target_size=(180,180))
    Image_Keras = keras.utils.img_to_array(Image_Keras)
    Image_Keras = keras.ops.expand_dims(Image_Keras, 0)
    predict = dnn_model.predict(Image_Keras)
    predicted_class=Classes[np.argmax(predict)]
    predictions.append(predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━